In [ ]:
import torch

In [2]:
L_pde = torch.tensor(0.1)
L_ic  = torch.tensor(0.01)
L_bc  = torch.tensor(0.001)

# Example PyTorch code (d=15): SoftAdapt-like weighting
loss_values = torch.tensor([L_pde.item(), L_ic.item(), L_bc.item()])
weights = torch.nn.functional.softmax(loss_values, dim=0)  # higher weights for larger losses
print(weights)
L_total = weights[0]*L_pde + weights[1]*L_ic + weights[2]*L_bc

tensor([0.3547, 0.3241, 0.3212])


In [3]:
import torch.nn as nn
class PINN(nn.Module):
    def __init__(self, D, layers=[64]):
        """
        D: input dimension (d spatial dims + 1 time dim)
        activation_fn: 'nn.Tanh', 'nn.SiLU'
        """
        super().__init__()

        net_layers = []
        for l1,l2 in zip(layers[:-1], layers[1:]):
            net_layers.append(nn.Linear(l1,l2))
            net_layers.append(nn.Tanh())

        self.net = nn.Sequential(
            nn.Linear(D, layers[0]), nn.Tanh(),
            *net_layers,
            nn.Linear(layers[-1], 1)
        )

    def forward(self, x):
        return self.net(x)

d = 2
D = d + 1
model = PINN(D, layers=[124,124])

In [57]:
X = torch.rand(10, D)
X.requires_grad = True
from main import compute_u_grad_u
u, grad_u = compute_u_grad_u(model, X)
R_pde = grad_u[:,-1:] * u + 10 * torch.sum(grad_u[:,:2]**2, dim=1).unsqueeze(dim=1) + grad_u[:,:2] @ torch.ones((d,1)) * u + u**2
L_pde = torch.mean(R_pde**2)
L_ic = torch.mean((u - (X[:,0]*(1-X[:,0])).unsqueeze(dim=1))**2)
L_bc = torch.mean(u**2)
print(L_pde.item(), L_ic.item(), L_bc.item())
# my idea:
w_pde = 1/L_pde.item()
w_ic = 1/L_ic.item()
w_bc = 1/L_bc.item()
torch.pi*torch.tensor([w_pde, w_ic, w_bc])/(w_pde+w_ic+w_bc)

0.006645950488746166 0.0867113247513771 0.008009887300431728


tensor([1.6480, 0.1263, 1.3673])

## GradNorm

In [53]:
# Example PyTorch code (d=10): GradNorm-inspired weighting
# Compute gradient norms for each loss term
grads_pde = torch.autograd.grad(L_pde, model.parameters(), retain_graph=True)
norm_pde = torch.sqrt(sum((g**2).sum() for g in grads_pde))
grads_ic = torch.autograd.grad(L_ic, model.parameters(), retain_graph=True)
norm_ic = torch.sqrt(sum((g**2).sum() for g in grads_ic))
grads_bc = torch.autograd.grad(L_bc, model.parameters(), retain_graph=True)
norm_bc = torch.sqrt(sum((g**2).sum() for g in grads_bc))
# Set weights to equalize norms (simple average target)
print(norm_pde, norm_ic, norm_bc)
avg_norm = (norm_pde + norm_ic + norm_bc) / 3.0
w_pde = (avg_norm / (norm_pde + 1e-8))
w_ic  = (avg_norm / (norm_ic  + 1e-8))
w_bc  = (avg_norm / (norm_bc  + 1e-8))
L_total = w_pde * L_pde + w_ic * L_ic + w_bc * L_bc
print(w_pde, w_ic, w_bc)

tensor(0.6101) tensor(1.7234) tensor(0.6160)
tensor(1.6114) tensor(0.5705) tensor(1.5960)


In [52]:
for g in grads_pde:
    print(g.shape)

torch.Size([124, 3])
torch.Size([124])
torch.Size([124, 124])
torch.Size([124])
torch.Size([1, 124])
torch.Size([1])


## SoftAdapt

In [59]:
# Trainable SoftAdapt weights (per-term here; extend to per-point)
alpha_pde = nn.Parameter(torch.tensor(1e-3))
alpha_bc = nn.Parameter(torch.tensor(1e-3))
optimizer = torch.optim.Adam(list(model.parameters()) + [alpha_pde, alpha_bc], lr=1e-3)